# R2/R3/R4 해당 생성 (Qwen2.5-7B-Instruct, 4bit)

**실행 전 준비물** (Google Drive `MyDrive/ade-project/`에 업로드):
- `unified.jsonl` (로컬 `research-project/data/unified.jsonl` — 원문이 포함되어 gitignore 대상이라 git clone에 안 따라옴)

**주의**: 이 노트북은 학생 모델 학습(train_qlora.ipynb)과 같은 세션에서 돌리지 말 것 (VRAM 부족). 끝나면 런타임을 재시작해 VRAM을 비우고 다음 단계로 넘어갈 것.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/ade-project'
import os
assert os.path.exists(f'{DRIVE_DIR}/unified.jsonl'), (
    f'{DRIVE_DIR}/unified.jsonl 없음 — 로컬 research-project/data/unified.jsonl을 '
    'Drive의 이 경로에 먼저 업로드할 것')

Mounted at /content/drive


In [3]:
REPO_URL = 'https://github.com/Gaeul5/Oracle_healthcare-bio_sLLM.git'

import os
if not os.path.exists('/content/repo'):
    !git clone -q {REPO_URL} /content/repo
else:
    !git -C /content/repo pull -q
%cd /content/repo/research-project

/content/repo/research-project


In [ ]:
!pip install -q -U vllm
!pip uninstall -y torchaudio -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.9/312.9 MB 5.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 97.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.9/184.9 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 108.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 15.5 MB/s eta 0:00:0000:01m00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.4/358.4 kB 33.4 MB/s eta 0:00:00
   ━━━━━━

## 1. 소량으로 속도/VRAM 확인 (먼저 실행)

전체 10,340건 × 3조건을 바로 돌리기 전에, 20건만으로 속도를 재고 전체 소요 시간을 가능해보자.

In [5]:
!python src/generate_teacher_data.py \
    --unified {DRIVE_DIR}/unified.jsonl \
    --out {DRIVE_DIR}/teacher_outputs.jsonl \
    --checkpoint-every 5 \
    --limit 20

전체 20건 / 이미 완료 0건 / 남은 20건
config.json: 100% 663/663 [00:00<00:00, 3.66MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 16.8MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 88.3MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 82.6MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 106MB/s]
model.safetensors.index.json: 100% 27.8k/27.8k [00:00<00:00, 88.2MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/3.56G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/11.3G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/15.2G [00:00<?, ?B/s]
Reconstructing (incomplete total...):  14% 2.20G/15.2G [00:18<00:57, 227MB/s, 26.7MB/s  ]
Reconstructing (incomplete total...):  15% 2.25G/15.2G [00:20<02:02, 106MB/s, 26.3MB/s  ]
Reconstructing (incomplete total...):  16% 2.36G/15.2G [00:20<01:36, 134MB/s, 25.5MB/s  ]
Reconstructin

위 출력의 "남은 예상 N분"을 보고 전체(10,320건 남음) 소요 시간을 가능해볼 것. 자유 Colab은 세션 시간 제한이 있으니, 오래 걸릴 것 같으면 여러 세션에 걸쳐 아래 셀을 반복 실행하면 된다 (이미 완료된 doc_id는 자동으로 건너뜀).

## 2. 전체 실행 (여러 세션에 걸쳐 재실행 가능 — 이미 완료된 doc_id는 건너뜀)

In [6]:
!python /content/repo/research-project/src/generate_teacher_data_vllm.py \
    --unified {DRIVE_DIR}/unified.jsonl \
    --out {DRIVE_DIR}/teacher_outputs.jsonl


전체 10340건 / 이미 완료 1520건 / 남은 8820건
INFO 08-13 02:01:41 [api_utils.py:273] non-default args: {'dtype': 'float16', 'max_model_len': 3072, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'quantization': 'awq', 'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ'}
INFO 08-13 02:01:58 [model.py:645] Resolved architecture: Qwen2ForCausalLM
INFO 08-13 02:01:58 [model.py:1883] Using max model len 3072
INFO 08-13 02:02:00 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.
model.safetensors.index.json: 100% 62.7k/62.7k [00:00<00:00, 110MB/s]
Parse safetensors files: 100% 2/2 [00:00<00:00,  2.19it/s]
INFO 08-13 02:02:02 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
generation_config.json: 100% 243/243 [00:00<00:00, 1.03MB/s]
(EngineCore pid=2447) INFO 08-13 02:02:11 [core.py:121] Initializing a V1 LLM engine (v0.27.1) with config: model='Qwen/Qwen2.5-7B-Instruct-AWQ', speculati

In [7]:
import json, sys, os
sys.path.insert(0, "/content/repo/research-project/src")
from formats import check_length_balance

out_path = f"{DRIVE_DIR}/teacher_outputs.jsonl"
records = [json.loads(l) for l in open(out_path, encoding="utf-8") if l.strip()]
print(f"완료된 문서 수: {len(records)} / 목표: 10340")

outputs_by_cond = {c: [r[c] for r in records] for c in ("R2", "R3", "R4")}
report = check_length_balance(outputs_by_cond)
print(json.dumps(report, indent=2, ensure_ascii=False))

bad = [c for c, v in report.items() if not v["ok"]]
print("문제 없음" if not bad else f"경고: {bad} 조건이 R2 대비 ±20% 범위를 벗어남")


완료된 문서 수: 10340 / 목표: 10340
{
  "R2": {
    "mean_tokens": 98.8,
    "ratio_to_R2": 1.0,
    "ok": true
  },
  "R3": {
    "mean_tokens": 98.2,
    "ratio_to_R2": 0.994,
    "ok": true
  },
  "R4": {
    "mean_tokens": 102.6,
    "ratio_to_R2": 1.038,
    "ok": true
  }
}
문제 없음
